# Module B Testing Notebook (Assignment 3)


This notebook is now the single place for Module B testing in this repository.
It covers:


- environment and preflight checks
- API smoke validation
- race-condition testing
- failure-simulation testing
- high-load stress testing
- expanded workload ranges across multiple profiles
- consolidated artifact export

In [5]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone


def detect_module_b_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "app").exists() and (cwd / "performance").exists():
        return cwd
    if (cwd / "Module_B" / "app").exists() and (cwd / "Module_B" / "performance").exists():
        return cwd / "Module_B"
    raise RuntimeError("Could not detect Module_B root. Open this notebook from Module_B or repo root.")


MODULE_B_ROOT = detect_module_b_root()
PERF_DIR = MODULE_B_ROOT / "performance"
PERF_SCRIPT = PERF_DIR / "run_module_b_concurrency_stress.py"


print(f"Module_B root: {MODULE_B_ROOT}")
print(f"Stress runner: {PERF_SCRIPT}")
if not PERF_SCRIPT.exists():
    raise FileNotFoundError(f"Missing stress runner script: {PERF_SCRIPT}")

Module_B root: C:\Users\Manoj\OneDrive\Acads\sem 6\Databases\DB_A3\Module_B
Stress runner: C:\Users\Manoj\OneDrive\Acads\sem 6\Databases\DB_A3\Module_B\performance\run_module_b_concurrency_stress.py


## Configuration


Set credentials and test target values here. Defaults match the seeded local setup used in this repo.
You can override values using environment variables before running the notebook.

In [6]:
BASE_URL = os.getenv("MODULE_B_BASE_URL", "http://127.0.0.1:8001")
USERNAMES = [
    u.strip()
    for u in os.getenv(
        "MODULE_B_USERNAMES",
        "rahul.sharma@iitgn.ac.in,priya.patel@iitgn.ac.in,ananya.singh@iitgn.ac.in,neha.desai@iitgn.ac.in,aditya.verma@iitgn.ac.in",
    ).split(",")
    if u.strip()
]
PASSWORD = os.getenv("MODULE_B_PASSWORD", "password123")
TARGET_MEMBER_ID = int(os.getenv("MODULE_B_TARGET_MEMBER_ID", "19"))
POST_ID = int(os.getenv("MODULE_B_POST_ID", "1"))


# Expanded testing range compared to the previous single-run setup.
RUN_EXTREME = False  # Set True to include a very heavy profile.
TEST_MATRIX = [
    {
        "name": "smoke",
        "race_requests": 200,
        "race_workers": 40,
        "failure_requests": 120,
        "failure_workers": 24,
        "stress_requests": 1000,
        "stress_workers": 80,
        "offset_window": 1500,
    },
    {
        "name": "medium",
        "race_requests": 500,
        "race_workers": 80,
        "failure_requests": 300,
        "failure_workers": 48,
        "stress_requests": 3000,
        "stress_workers": 120,
        "offset_window": 2500,
    },
    {
        "name": "high",
        "race_requests": 1000,
        "race_workers": 120,
        "failure_requests": 600,
        "failure_workers": 80,
        "stress_requests": 5000,
        "stress_workers": 160,
        "offset_window": 4000,
    },
]


if RUN_EXTREME:
    TEST_MATRIX.append(
        {
            "name": "extreme",
            "race_requests": 2000,
            "race_workers": 240,
            "failure_requests": 1200,
            "failure_workers": 120,
            "stress_requests": 10000,
            "stress_workers": 300,
            "offset_window": 8000,
        }
    )


print("Base URL:", BASE_URL)
print("Users:", len(USERNAMES), USERNAMES)
print("Target member:", TARGET_MEMBER_ID, "Post:", POST_ID)
print("Profiles:", [p["name"] for p in TEST_MATRIX])

Base URL: http://127.0.0.1:8001
Users: 5 ['rahul.sharma@iitgn.ac.in', 'priya.patel@iitgn.ac.in', 'ananya.singh@iitgn.ac.in', 'neha.desai@iitgn.ac.in', 'aditya.verma@iitgn.ac.in']
Target member: 19 Post: 1
Profiles: ['smoke', 'medium', 'high']


In [7]:
import urllib.error
import urllib.request


def api_request(method: str, path: str, payload: dict | None = None, token: str | None = None, timeout_s: int = 20) -> dict:
    data = None
    headers = {"Content-Type": "application/json"}
    if token:
        headers["session-token"] = token
    if payload is not None:
        data = json.dumps(payload).encode("utf-8")


    req = urllib.request.Request(
        f"{BASE_URL.rstrip('/')}{path}",
        data=data,
        headers=headers,
        method=method,
    )


    try:
        with urllib.request.urlopen(req, timeout=timeout_s) as resp:
            body = resp.read().decode("utf-8")
            return {
                "ok": 200 <= resp.status < 300,
                "status": resp.status,
                "body": json.loads(body) if body else {},
            }
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8") if exc.fp is not None else ""
        try:
            parsed = json.loads(body) if body else {}
        except json.JSONDecodeError:
            parsed = {"detail": body}
        return {"ok": False, "status": exc.code, "body": parsed}


def preflight_checks() -> dict:
    result = {
        "runner_help_ok": False,
        "api_reachable": False,
        "auth_ok": False,
        "feed_ok": False,
        "ready_for_matrix": False,
        "sample_user": USERNAMES[0] if USERNAMES else "",
        "message": "",
    }


    help_proc = subprocess.run(
        [sys.executable, str(PERF_SCRIPT), "--help"],
        cwd=str(MODULE_B_ROOT),
        capture_output=True,
        text=True,
    )
    if help_proc.returncode != 0:
        result["message"] = f"Stress runner check failed: {help_proc.stderr.strip()}"
        return result


    result["runner_help_ok"] = True


    try:
        login = api_request(
            "POST",
            "/login",
            payload={"username": USERNAMES[0], "password": PASSWORD},
        )
        result["api_reachable"] = True
    except Exception as exc:  # noqa: BLE001
        result["message"] = f"API not reachable at {BASE_URL}: {exc}"
        return result


    result["login_status"] = login.get("status", 0)
    if not login["ok"]:
        result["message"] = f"Login preflight failed: {login}"
        return result


    token = login["body"].get("session_token")
    if not token:
        result["message"] = "Login preflight did not return session_token"
        return result


    result["auth_ok"] = True


    feed = api_request("GET", "/posts?limit=5&offset=0", token=token)
    result["feed_status"] = feed.get("status", 0)
    if not feed["ok"]:
        result["message"] = f"Feed preflight failed: {feed}"
        return result


    result["feed_ok"] = True
    result["ready_for_matrix"] = True
    result["message"] = "Preflight checks passed."
    return result


preflight = preflight_checks()
preflight

{'runner_help_ok': True,
 'api_reachable': True,
 'auth_ok': True,
 'feed_ok': True,
 'ready_for_matrix': True,
 'sample_user': 'rahul.sharma@iitgn.ac.in',
 'message': 'Preflight checks passed.',
 'login_status': 200,
 'feed_status': 200}

## Expanded Test Matrix Execution


This runs all profiles in `TEST_MATRIX` and writes one JSON report per profile to the `performance` folder.

In [8]:
def extract_json_block(text: str) -> dict:
    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError(f"Could not parse JSON block from output:\n{text}")
    return json.loads(text[start:end + 1])


def run_profile(profile: dict) -> dict:
    output_path = PERF_DIR / f"module_b_concurrency_report_{profile['name']}.json"
    cmd = [
        sys.executable,
        str(PERF_SCRIPT),
        "--base-url", BASE_URL,
        "--usernames", ",".join(USERNAMES),
        "--password", PASSWORD,
        "--target-member-id", str(TARGET_MEMBER_ID),
        "--post-id", str(POST_ID),
        "--race-requests", str(profile["race_requests"]),
        "--race-workers", str(profile["race_workers"]),
        "--failure-requests", str(profile["failure_requests"]),
        "--failure-workers", str(profile["failure_workers"]),
        "--stress-requests", str(profile["stress_requests"]),
        "--stress-workers", str(profile["stress_workers"]),
        "--offset-window", str(profile["offset_window"]),
        "--output", str(output_path),
    ]


    t0 = time.perf_counter()
    proc = subprocess.run(
        cmd,
        cwd=str(MODULE_B_ROOT),
        capture_output=True,
        text=True,
    )
    elapsed_s = round(time.perf_counter() - t0, 3)


    result = {
        "profile": profile["name"],
        "config": profile,
        "elapsed_s": elapsed_s,
        "return_code": proc.returncode,
        "stdout": proc.stdout,
        "stderr": proc.stderr,
        "output_path": str(output_path),
    }


    if proc.returncode != 0:
        result["status"] = "runner_error"
        return result


    summary = extract_json_block(proc.stdout)
    result["status"] = "completed"
    result["summary"] = summary


    if output_path.exists():
        result["report"] = json.loads(output_path.read_text(encoding="utf-8"))
    else:
        result["status"] = "missing_report"


    return result


if not preflight.get("ready_for_matrix", False):
    raise RuntimeError(
        f"Preflight not ready. Start API and ensure sample users exist. Details: {preflight.get('message', '')}"
    )


matrix_results = []
for profile in TEST_MATRIX:
    print(f"Running profile: {profile['name']}")
    r = run_profile(profile)
    matrix_results.append(r)
    print(f"  status={r['status']}, rc={r['return_code']}, elapsed={r['elapsed_s']}s")


matrix_results

Running profile: smoke
  status=completed, rc=0, elapsed=8.905s
Running profile: medium
  status=completed, rc=0, elapsed=20.873s
Running profile: high
  status=completed, rc=0, elapsed=44.732s


[{'profile': 'smoke',
  'config': {'name': 'smoke',
   'race_requests': 200,
   'race_workers': 40,
   'failure_requests': 120,
   'failure_workers': 24,
   'stress_requests': 1000,
   'stress_workers': 80,
   'offset_window': 1500},
  'elapsed_s': 8.905,
  'return_code': 0,
  'stdout': '{\n  "overall_pass": true,\n  "race_passed": true,\n  "failure_simulation_passed": true,\n  "stress_passed": true,\n  "output": "C:\\\\Users\\\\Manoj\\\\OneDrive\\\\Acads\\\\sem 6\\\\Databases\\\\DB_A3\\\\Module_B\\\\performance\\\\module_b_concurrency_report_smoke.json"\n}\n',
  'stderr': '',
  'output_path': 'C:\\Users\\Manoj\\OneDrive\\Acads\\sem 6\\Databases\\DB_A3\\Module_B\\performance\\module_b_concurrency_report_smoke.json',
  'status': 'completed',
  'summary': {'overall_pass': True,
   'race_passed': True,
   'failure_simulation_passed': True,
   'stress_passed': True,
   'output': 'C:\\Users\\Manoj\\OneDrive\\Acads\\sem 6\\Databases\\DB_A3\\Module_B\\performance\\module_b_concurrency_report_

In [9]:
def pass_label(value: bool) -> str:
    return "PASS" if value else "FAIL"


rows = []
for r in matrix_results:
    if r["status"] != "completed":
        rows.append({
            "profile": r["profile"],
            "status": r["status"],
            "overall": False,
            "race": False,
            "failure": False,
            "stress": False,
            "throughput": 0.0,
            "stress_p95_ms": 0.0,
        })
        continue


    report = r["report"]
    race = bool(report["race_follow_test"]["race_passed"])
    failure = bool(report["failure_simulation"]["failure_simulation_passed"])
    stress = bool(report["stress_reads"]["stress_passed"])
    overall = bool(report["overall_pass"])

    rows.append({
        "profile": r["profile"],
        "status": r["status"],
        "overall": overall,
        "race": race,
        "failure": failure,
        "stress": stress,
        "throughput": float(report["stress_reads"]["throughput_req_per_s"]),
        "stress_p95_ms": float(report["stress_reads"]["latency"]["p95_ms"]),
    })


print("\nProfile Summary")
print("-" * 92)
print(f"{'profile':<12} {'status':<12} {'overall':<8} {'race':<8} {'failure':<8} {'stress':<8} {'throughput':>12} {'p95(ms)':>10}")
print("-" * 92)
for row in rows:
    print(
        f"{row['profile']:<12} {row['status']:<12} {pass_label(row['overall']):<8} {pass_label(row['race']):<8} "
        f"{pass_label(row['failure']):<8} {pass_label(row['stress']):<8} {row['throughput']:>12.3f} {row['stress_p95_ms']:>10.3f}"
    )
print("-" * 92)


notebook_overall_pass = all(r["status"] == "completed" and r["overall"] for r in rows)
print("Notebook overall pass:", notebook_overall_pass)


if not notebook_overall_pass:
    raise AssertionError("One or more profiles failed. Inspect matrix_results for details.")


Profile Summary
--------------------------------------------------------------------------------------------
profile      status       overall  race     failure  stress     throughput    p95(ms)
--------------------------------------------------------------------------------------------
smoke        completed    PASS     PASS     PASS     PASS          239.499    370.633
medium       completed    PASS     PASS     PASS     PASS          221.402    583.748
high         completed    PASS     PASS     PASS     PASS          178.665   1062.457
--------------------------------------------------------------------------------------------
Notebook overall pass: True


In [10]:
matrix_artifact = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "base_url": BASE_URL,
    "users": USERNAMES,
    "profile_count": len(TEST_MATRIX),
    "run_extreme": RUN_EXTREME,
    "results": matrix_results,
    "rows": rows,
    "notebook_overall_pass": notebook_overall_pass,
}


matrix_output_path = PERF_DIR / "module_b_notebook_test_matrix_results.json"
matrix_output_path.write_text(json.dumps(matrix_artifact, indent=2), encoding="utf-8")
print(f"Saved notebook matrix artifact: {matrix_output_path}")

Saved notebook matrix artifact: C:\Users\Manoj\OneDrive\Acads\sem 6\Databases\DB_A3\Module_B\performance\module_b_notebook_test_matrix_results.json


## Notes


- Run cells top to bottom.
- Keep `RUN_EXTREME = False` for normal grading runs.
- Set `RUN_EXTREME = True` only when you want a larger stress envelope.
- Generated per-profile reports are saved in `Module_B/performance/`.
- A consolidated notebook artifact is saved as `module_b_notebook_test_matrix_results.json`.